In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from drn import GLM, CANN, MDN, DDR, DRN, preprocess_data, drn_cutpoints, train

from generate_synthetic_dataset import generate_synthetic_gamma

from analysis_utils import (
    process_data_with_std,
    generate_latex_table_more_runs,
    plot_metrics_grid,
    get_nll_crps_rmse_ql,
)

torch.set_num_threads(1)

In [ ]:
# Create a shared large test set (without standardising - it will need to be adjusted for each model later).
x_test_raw_shared, y_test_raw_shared, _, _ = generate_synthetic_gamma(
    10_000, seed=131313
)
x_test_raw_shared = pd.DataFrame(x_test_raw_shared, columns=["X_1", "X_2"])
Y_test_shared = torch.Tensor(y_test_raw_shared.values).flatten()

In [ ]:
np.random.seed(32052025)

NUM_DATASET_SEEDS = 20
dataset_seeds = [
    int(s) for s in np.random.randint(0, 2**32 - 1, size=NUM_DATASET_SEEDS)
]

sizes = [1_000, 3_000, 6_000]
datasets = {}

for i, dataset_seed in enumerate(dataset_seeds):
    # For each seed, make the largest dataset size
    features, target, _, _ = generate_synthetic_gamma(
        int(sizes[-1] * 0.8), seed=dataset_seed
    )

    x_train_raw, x_val_raw, y_train_raw, y_val_raw = train_test_split(
        features, target, test_size=0.25, random_state=42, shuffle=True
    )

    # Then subset it to the smaller sizes
    for size in sizes:
        x_train_size = x_train_raw[: int(0.6 * size)]
        y_train = y_train_raw[: int(0.6 * size)]
        x_val_size = x_val_raw[: int(0.2 * size)]
        y_val = y_val_raw[: int(0.2 * size)]

        x_train, x_val, x_test_shared, _, _ = preprocess_data(
            x_train_size,
            x_val_size,
            x_test_raw_shared,
            num_features=["X_1", "X_2"],
            cat_features=[],
            num_standard=True,
        )

        datasets[(size, i)] = (x_train, y_train, x_val, y_val, x_test_shared)

# Table D.6

In [ ]:
@dataclass
class TrainParams:
    proportion: float
    hidden_size: int
    dropout_rate: float
    num_hidden_layers: int
    lr: float
    batch_size: int
    patience: int
    kl_alpha: float


SIZE_TO_PARAMS: dict[int, TrainParams] = {
    1000: TrainParams(
        proportion=0.2,
        hidden_size=128,
        dropout_rate=0.5,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
    3000: TrainParams(
        proportion=0.1,
        hidden_size=256,
        dropout_rate=0.4,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
    6000: TrainParams(
        proportion=0.1,
        hidden_size=512,
        dropout_rate=0.3,
        num_hidden_layers=2,
        lr=1e-3 / 4,
        batch_size=128,
        patience=100,
        kl_alpha=0.05,
    ),
}

distribution = "gamma"

In [ ]:
make_empty_dict = lambda: {1000: [], 3000: [], 6000: []}
glm_gamma_dict_nll = make_empty_dict()
glm_ig_dict_nll = make_empty_dict()
glm_gamma_dict_crps = make_empty_dict()
glm_ig_dict_crps = make_empty_dict()
glm_gamma_dict_rmse = make_empty_dict()
glm_ig_dict_rmse = make_empty_dict()
glm_gamma_dict_ql90 = make_empty_dict()
glm_ig_dict_ql90 = make_empty_dict()

glm_gamma_null_dict_nll = make_empty_dict()
glm_ig_null_dict_nll = make_empty_dict()
glm_gamma_null_dict_crps = make_empty_dict()
glm_ig_null_dict_crps = make_empty_dict()
glm_gamma_null_dict_rmse = make_empty_dict()
glm_ig_null_dict_rmse = make_empty_dict()
glm_gamma_null_dict_ql90 = make_empty_dict()
glm_ig_null_dict_ql90 = make_empty_dict()

glm_gamma_empty_dict_nll = make_empty_dict()
glm_ig_empty_dict_nll = make_empty_dict()
glm_gamma_empty_dict_crps = make_empty_dict()
glm_ig_empty_dict_crps = make_empty_dict()
glm_gamma_empty_dict_rmse = make_empty_dict()
glm_ig_empty_dict_rmse = make_empty_dict()
glm_gamma_empty_dict_ql90 = make_empty_dict()
glm_ig_empty_dict_ql90 = make_empty_dict()

In [ ]:
for size in [1000, 3000, 6000]:
    params = SIZE_TO_PARAMS[size]
    proportion = params.proportion
    hidden_size = params.hidden_size
    dropout_rate = params.dropout_rate
    num_hidden_layers = params.num_hidden_layers
    lr = params.lr
    batch_size = params.batch_size
    patience = params.patience
    kl_alpha = params.kl_alpha

    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma", default=True)
        glm_ig_empty = GLM(p=2, distribution="inversegaussian", default=True)

        nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
            get_nll_crps_rmse_ql(
                models=[
                    glm_gamma_null,
                    glm_gamma_empty,
                    glm_ig_null,
                    glm_ig_empty,
                    glm_gamma,
                    glm_ig,
                ],
                names=[
                    "GLM_GA_NULL",
                    "GLM_GA_EMPTY",
                    "GLM_IG_NULL",
                    "GLM_IG_EMPTY",
                    "GLM_GA",
                    "GLM_IG",
                ],
                X_test_data=X_test_shared,
                Y_test_data=Y_test_shared,
                y_train=y_train,
            )
        )

        glm_gamma_null_dict_nll[size].append(nll_test_dict["GLM_GA_NULL"].item())
        glm_gamma_null_dict_crps[size].append(crps_test_dict["GLM_GA_NULL"].item())
        glm_gamma_null_dict_rmse[size].append(rmse_test_dict["GLM_GA_NULL"].item())
        glm_gamma_null_dict_ql90[size].append(ql_90_test_dict["GLM_GA_NULL"].item())

        glm_ig_null_dict_nll[size].append(nll_test_dict["GLM_IG_NULL"].item())
        glm_ig_null_dict_crps[size].append(crps_test_dict["GLM_IG_NULL"].item())
        glm_ig_null_dict_rmse[size].append(rmse_test_dict["GLM_IG_NULL"].item())
        glm_ig_null_dict_ql90[size].append(ql_90_test_dict["GLM_IG_NULL"].item())

        glm_gamma_empty_dict_nll[size].append(nll_test_dict["GLM_GA_EMPTY"].item())
        glm_gamma_empty_dict_crps[size].append(crps_test_dict["GLM_GA_EMPTY"].item())
        glm_gamma_empty_dict_rmse[size].append(rmse_test_dict["GLM_GA_EMPTY"].item())
        glm_gamma_empty_dict_ql90[size].append(ql_90_test_dict["GLM_GA_EMPTY"].item())

        glm_ig_empty_dict_nll[size].append(nll_test_dict["GLM_IG_EMPTY"].item())
        glm_ig_empty_dict_crps[size].append(crps_test_dict["GLM_IG_EMPTY"].item())
        glm_ig_empty_dict_rmse[size].append(rmse_test_dict["GLM_IG_EMPTY"].item())
        glm_ig_empty_dict_ql90[size].append(ql_90_test_dict["GLM_IG_EMPTY"].item())

        glm_gamma_dict_nll[size].append(nll_test_dict["GLM_GA"].item())
        glm_gamma_dict_crps[size].append(crps_test_dict["GLM_GA"].item())
        glm_gamma_dict_rmse[size].append(rmse_test_dict["GLM_GA"].item())
        glm_gamma_dict_ql90[size].append(ql_90_test_dict["GLM_GA"].item())

        glm_ig_dict_nll[size].append(nll_test_dict["GLM_IG"].item())
        glm_ig_dict_crps[size].append(crps_test_dict["GLM_IG"].item())
        glm_ig_dict_rmse[size].append(rmse_test_dict["GLM_IG"].item())
        glm_ig_dict_ql90[size].append(ql_90_test_dict["GLM_IG"].item())

In [ ]:
data_dicts = {
    "NLL": {
        "GLM_GA": glm_gamma_dict_nll,
        "GLM_IG": glm_ig_dict_nll,
        "GLM_GA_NULL": glm_gamma_null_dict_nll,
        "GLM_IG_NULL": glm_ig_null_dict_nll,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_nll,
        "GLM_IG_EMPTY": glm_ig_empty_dict_nll,
    },
    "CRPS": {
        "GLM_GA": glm_gamma_dict_crps,
        "GLM_IG": glm_ig_dict_crps,
        "GLM_GA_NULL": glm_gamma_null_dict_crps,
        "GLM_IG_NULL": glm_ig_null_dict_crps,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_crps,
        "GLM_IG_EMPTY": glm_ig_empty_dict_crps,
    },
    "RMSE": {
        "GLM_GA": glm_gamma_dict_rmse,
        "GLM_IG": glm_ig_dict_rmse,
        "GLM_GA_NULL": glm_gamma_null_dict_rmse,
        "GLM_IG_NULL": glm_ig_null_dict_rmse,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_rmse,
        "GLM_IG_EMPTY": glm_ig_empty_dict_rmse,
    },
    "QL90": {
        "GLM_GA": glm_gamma_dict_ql90,
        "GLM_IG": glm_ig_dict_ql90,
        "GLM_GA_NULL": glm_gamma_null_dict_ql90,
        "GLM_IG_NULL": glm_ig_null_dict_ql90,
        "GLM_GA_EMPTY": glm_gamma_empty_dict_ql90,
        "GLM_IG_EMPTY": glm_ig_empty_dict_ql90,
    },
}

metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = [
    "GLM_GA",
    "GLM_IG",
    "GLM_GA_NULL",
    "GLM_IG_NULL",
    "GLM_GA_EMPTY",
    "GLM_IG_EMPTY",
]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(data_dicts, metrics, models, keys=[1000, 3000, 6000])

# Top Panel of Table D.7

In [ ]:
drn_gamma_dict_nll = make_empty_dict()
drn_ig_dict_nll = make_empty_dict()
drn_ig_kl_small_dict_nll = make_empty_dict()
mdn_dict_nll = make_empty_dict()
ddr_dict_nll = make_empty_dict()

drn_gamma_dict_crps = make_empty_dict()
drn_ig_dict_crps = make_empty_dict()
drn_ig_kl_small_dict_crps = make_empty_dict()
mdn_dict_crps = make_empty_dict()
ddr_dict_crps = make_empty_dict()

drn_gamma_dict_rmse = make_empty_dict()
drn_ig_dict_rmse = make_empty_dict()
drn_ig_kl_small_dict_rmse = make_empty_dict()
mdn_dict_rmse = make_empty_dict()
ddr_dict_rmse = make_empty_dict()

drn_gamma_dict_ql90 = make_empty_dict()
drn_ig_dict_ql90 = make_empty_dict()
drn_ig_kl_small_dict_ql90 = make_empty_dict()
mdn_dict_ql90 = make_empty_dict()
ddr_dict_ql90 = make_empty_dict()

In [ ]:
for size in [1000, 3000, 6000]:
    params = SIZE_TO_PARAMS[size]
    proportion = params.proportion
    hidden_size = params.hidden_size
    dropout_rate = params.dropout_rate
    num_hidden_layers = params.num_hidden_layers
    lr = params.lr
    batch_size = params.batch_size
    patience = params.patience
    kl_alpha = params.kl_alpha

    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

        print(f"Size: {size}; Seed: {i}")
        print(
            "-----------------------------------------------------------------------------------"
        )

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        cutpoints_DRN = drn_cutpoints(
            c_0=(
                np.min(Y_train.detach().numpy()) * 1.1
                if np.min(Y_train.detach().numpy()) < 0
                else 0.0
            ),
            c_K=20,
            proportion=proportion,
            y=Y_train.detach().numpy(),
            min_obs=3,
        )

        torch.manual_seed(23)
        drn_gamma = DRN(
            glm=glm_gamma,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_gamma,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=50,
        )
        drn_gamma.eval()

        torch.manual_seed(23)
        drn_ig = DRN(
            glm=glm_ig,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_ig,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig.eval()

        torch.manual_seed(23)
        drn_ig_small_kl = DRN(
            glm=glm_ig,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="forwards",
        )

        train(
            model=drn_ig_small_kl,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_ig_small_kl.eval()

        torch.manual_seed(23)
        ddr = DDR(
            x_train.shape[1],
            cutpoints_DRN,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )
        train(
            ddr,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            lr=lr,
            batch_size=batch_size,
            log_interval=100,
            patience=patience,
            epochs=5000,
        )
        ddr.eval()

        torch.manual_seed(23)
        mdn = MDN(
            X_train.shape[1],
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
            num_components=5,
            distribution=distribution,
        )

        torch.manual_seed(23)
        train(
            mdn,
            train_dataset,
            val_dataset,
            lr=lr,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            log_interval=100,
        )
        mdn.eval()

        nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
            get_nll_crps_rmse_ql(
                models=[drn_ig, drn_ig_small_kl, ddr, mdn, drn_gamma],
                names=["DRN_IG", "DRN_IG_KL_SMALL", "DDR", "MDN", "DRN_GA"],
                X_test_data=X_test_shared,
                Y_test_data=Y_test_shared,
                y_train=y_train,
            )
        )

        drn_ig_dict_nll[size].append(nll_test_dict["DRN_IG"].item())
        drn_ig_dict_crps[size].append(crps_test_dict["DRN_IG"].item())
        drn_ig_dict_rmse[size].append(rmse_test_dict["DRN_IG"].item())
        drn_ig_dict_ql90[size].append(ql_90_test_dict["DRN_IG"].item())

        drn_ig_kl_small_dict_nll[size].append(nll_test_dict["DRN_IG_KL_SMALL"].item())
        drn_ig_kl_small_dict_crps[size].append(crps_test_dict["DRN_IG_KL_SMALL"].item())
        drn_ig_kl_small_dict_rmse[size].append(rmse_test_dict["DRN_IG_KL_SMALL"].item())
        drn_ig_kl_small_dict_ql90[size].append(
            ql_90_test_dict["DRN_IG_KL_SMALL"].item()
        )

        ddr_dict_nll[size].append(nll_test_dict["DDR"].item())
        ddr_dict_crps[size].append(crps_test_dict["DDR"].item())
        ddr_dict_rmse[size].append(rmse_test_dict["DDR"].item())
        ddr_dict_ql90[size].append(ql_90_test_dict["DDR"].item())

        mdn_dict_nll[size].append(nll_test_dict["MDN"].item())
        mdn_dict_crps[size].append(crps_test_dict["MDN"].item())
        mdn_dict_rmse[size].append(rmse_test_dict["MDN"].item())
        mdn_dict_ql90[size].append(ql_90_test_dict["MDN"].item())

        drn_gamma_dict_nll[size].append(nll_test_dict["DRN_GA"].item())
        drn_gamma_dict_crps[size].append(crps_test_dict["DRN_GA"].item())
        drn_gamma_dict_rmse[size].append(rmse_test_dict["DRN_GA"].item())
        drn_gamma_dict_ql90[size].append(ql_90_test_dict["DRN_GA"].item())

In [ ]:
# Define evaluation dictionaries for automation
nll_test_dict = {
    "DRN_IG": drn_ig_dict_nll,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_nll,
    "DDR": ddr_dict_nll,
    "MDN": mdn_dict_nll,
    "DRN_GA": drn_gamma_dict_nll,
}

crps_test_dict = {
    "DRN_IG": drn_ig_dict_crps,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_crps,
    "DDR": ddr_dict_crps,
    "MDN": mdn_dict_crps,
    "DRN_GA": drn_gamma_dict_crps,
}

rmse_test_dict = {
    "DRN_IG": drn_ig_dict_rmse,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_rmse,
    "DDR": ddr_dict_rmse,
    "MDN": mdn_dict_rmse,
    "DRN_GA": drn_gamma_dict_rmse,
}

ql_90_test_dict = {
    "DRN_IG": drn_ig_dict_ql90,
    "DRN_IG_KL_SMALL": drn_ig_kl_small_dict_ql90,
    "DDR": ddr_dict_ql90,
    "MDN": mdn_dict_ql90,
    "DRN_GA": drn_gamma_dict_ql90,
}

data_dicts = {
    "NLL": nll_test_dict,
    "CRPS": crps_test_dict,
    "RMSE": rmse_test_dict,
    "QL90": ql_90_test_dict,
}


metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = ["DRN_IG", "DRN_IG_KL_SMALL", "DDR", "MDN", "DRN_GA"]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(data_dicts, metrics, models, keys=[1000, 3000, 6000])

# Bottom Panel of Table D.7

In [ ]:
drn_gamma_backward_null_nll = make_empty_dict()
drn_gamma_backward_empty_nll = make_empty_dict()
cann_gamma_null_nll = make_empty_dict()
cann_gamma_empty_nll = make_empty_dict()

drn_gamma_backward_null_crps = make_empty_dict()
drn_gamma_backward_empty_crps = make_empty_dict()
cann_gamma_null_crps = make_empty_dict()
cann_gamma_empty_crps = make_empty_dict()

drn_gamma_backward_null_rmse = make_empty_dict()
drn_gamma_backward_empty_rmse = make_empty_dict()
cann_gamma_null_rmse = make_empty_dict()
cann_gamma_empty_rmse = make_empty_dict()

drn_gamma_backward_null_ql90 = make_empty_dict()
drn_gamma_backward_empty_ql90 = make_empty_dict()
cann_gamma_null_ql90 = make_empty_dict()
cann_gamma_empty_ql90 = make_empty_dict()

In [ ]:
for size in [1000, 3000, 6000]:
    # PL NB: size=1000 Proportion was *2, and in all kl_alpha was 0.01
    params = SIZE_TO_PARAMS[size]
    proportion = params.proportion
    hidden_size = params.hidden_size
    dropout_rate = params.dropout_rate
    num_hidden_layers = params.num_hidden_layers
    lr = params.lr
    batch_size = params.batch_size
    patience = params.patience
    kl_alpha = params.kl_alpha

    for i in range(NUM_DATASET_SEEDS):
        x_train, y_train, x_val, y_val, x_test_shared = datasets[(size, i)]
        X_train = torch.Tensor(x_train.values)
        X_val = torch.Tensor(x_val.values)
        X_test_shared = torch.Tensor(x_test_shared.values)
        Y_train = torch.Tensor(y_train.values).flatten()
        Y_val = torch.Tensor(y_val.values).flatten()

        train_dataset = torch.utils.data.TensorDataset(X_train, Y_train)
        val_dataset = torch.utils.data.TensorDataset(X_val, Y_val)

        print(f"Size: {size}; Seed: {i}")
        print(
            "-----------------------------------------------------------------------------------"
        )

        glm_gamma = GLM.from_statsmodels(X_train, Y_train, distribution="gamma")
        glm_gamma.eval()

        glm_ig = GLM.from_statsmodels(X_train, Y_train, distribution="inversegaussian")
        glm_ig.eval()

        glm_gamma_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="gamma", null_model=True
        )
        glm_ig_null = GLM.from_statsmodels(
            X_train, Y_train, distribution="inversegaussian", null_model=True
        )

        glm_gamma_empty = GLM(p=2, distribution="gamma", default=True)
        glm_ig_empty = GLM(p=2, distribution="inversegaussian", default=True)

        cutpoints_DRN = drn_cutpoints(
            c_0=(
                np.min(Y_train.detach().numpy()) * 1.1
                if np.min(Y_train.detach().numpy()) < 0
                else 0.0
            ),
            c_K=20,  # np.max(Y_train.detach().numpy()) * 1.1,
            proportion=proportion,
            y=Y_train.detach().numpy(),
            min_obs=3,
        )

        torch.manual_seed(23)
        drn_gamma_null = DRN(
            glm=glm_gamma_null,
            cutpoints=cutpoints_DRN,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="backward",
        )

        train(
            model=drn_gamma_null,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_null.eval()

        torch.manual_seed(23)
        drn_gamma_empty = DRN(
            cutpoints=cutpoints_DRN,
            glm=glm_gamma_empty,
            hidden_size=hidden_size,
            num_hidden_layers=num_hidden_layers,
            baseline_start=False,
            dropout_rate=dropout_rate,
            kl_alpha=kl_alpha,
            dv_alpha=0.0,
            loss_metric="jbce",
            kl_direction="backward",
        )
        train(
            model=drn_gamma_empty,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            batch_size=batch_size,
            epochs=5000,
            patience=patience,
            lr=lr,
            print_details=True,
            log_interval=100,
        )
        drn_gamma_empty.eval()

        torch.manual_seed(23)
        cann_gamma_null = CANN(
            glm_gamma_null,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )

        train(
            cann_gamma_null,
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_null.update_dispersion(X_train, Y_train)
        cann_gamma_null.eval()

        torch.manual_seed(23)
        cann_gamma_empty = CANN(
            glm_gamma_empty,
            num_hidden_layers=num_hidden_layers,
            hidden_size=hidden_size,
            dropout_rate=dropout_rate,
        )

        torch.manual_seed(23)
        train(
            cann_gamma_empty,
            train_dataset,
            val_dataset,
            epochs=5000,
            lr=lr,
            patience=patience,
            batch_size=batch_size,
            log_interval=100,
        )
        cann_gamma_empty.update_dispersion(X_train, Y_train)
        cann_gamma_empty.eval()

        nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
            get_nll_crps_rmse_ql(
                models=[
                    drn_gamma_empty,
                    drn_gamma_null,
                    cann_gamma_empty,
                    cann_gamma_null,
                ],
                names=["DRN_GA_EMPTY", "DRN_GA_NULL", "CANN_GA_EMPTY", "CANN_GA_NULL"],
                X_test_data=X_test_shared,
                Y_test_data=Y_test_shared,
                y_train=y_train,
            )
        )

        drn_gamma_backward_null_nll[size].append(nll_test_dict["DRN_GA_NULL"].item())
        drn_gamma_backward_null_crps[size].append(crps_test_dict["DRN_GA_NULL"].item())
        drn_gamma_backward_null_rmse[size].append(rmse_test_dict["DRN_GA_NULL"].item())
        drn_gamma_backward_null_ql90[size].append(ql_90_test_dict["DRN_GA_NULL"].item())

        drn_gamma_backward_empty_nll[size].append(nll_test_dict["DRN_GA_EMPTY"].item())
        drn_gamma_backward_empty_crps[size].append(
            crps_test_dict["DRN_GA_EMPTY"].item()
        )
        drn_gamma_backward_empty_rmse[size].append(
            rmse_test_dict["DRN_GA_EMPTY"].item()
        )
        drn_gamma_backward_empty_ql90[size].append(
            ql_90_test_dict["DRN_GA_EMPTY"].item()
        )

        cann_gamma_null_nll[size].append(nll_test_dict["CANN_GA_NULL"].item())
        cann_gamma_null_crps[size].append(crps_test_dict["CANN_GA_NULL"].item())
        cann_gamma_null_rmse[size].append(rmse_test_dict["CANN_GA_NULL"].item())
        cann_gamma_null_ql90[size].append(ql_90_test_dict["CANN_GA_NULL"].item())

        cann_gamma_empty_nll[size].append(nll_test_dict["CANN_GA_EMPTY"].item())
        cann_gamma_empty_crps[size].append(crps_test_dict["CANN_GA_EMPTY"].item())
        cann_gamma_empty_rmse[size].append(rmse_test_dict["CANN_GA_EMPTY"].item())
        cann_gamma_empty_ql90[size].append(ql_90_test_dict["CANN_GA_EMPTY"].item())

In [ ]:
data_dicts = {
    "NLL": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_nll,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_nll,
        "CANN_GA_NULL": cann_gamma_null_nll,
        "CANN_GA_BIASED": cann_gamma_empty_nll,
    },
    "CRPS": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_crps,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_crps,
        "CANN_GA_NULL": cann_gamma_null_crps,
        "CANN_GA_BIASED": cann_gamma_empty_crps,
    },
    "RMSE": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_rmse,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_rmse,
        "CANN_GA_NULL": cann_gamma_null_rmse,
        "CANN_GA_BIASED": cann_gamma_empty_rmse,
    },
    "QL90": {
        "DRN_GA_NULL (REVERSE KL)": drn_gamma_backward_null_ql90,
        "DRN_GA_BIASED (REVERSE KL)": drn_gamma_backward_empty_ql90,
        "CANN_GA_NULL": cann_gamma_null_ql90,
        "CANN_GA_BIASED": cann_gamma_empty_ql90,
    },
}

metrics = ["NLL", "CRPS", "RMSE", "QL90"]
models = [
    "CANN_GA_BIASED",
    "CANN_GA_NULL",
    "DRN_GA_BIASED (REVERSE KL)",
    "DRN_GA_NULL (REVERSE KL)",
]

plot_metrics_grid(
    data_dicts, metrics, models, process_data_with_std, keys=[1000, 3000, 6000]
)
generate_latex_table_more_runs(data_dicts, metrics, models, keys=[1000, 3000, 6000])